In [10]:
# ========Setup & Imports =========
import sys
from pathlib import Path
import re
from collections import Counter
import pandas as pd

root_path = Path.cwd().parent if Path.cwd().name in ["notebooks"] else Path.cwd()
if str(root_path) not in sys.path:
    sys.path.append(str(root_path))

from src.client import complete_gap


In [11]:
# ========Dataset Definition =========
questions = [
    {
        "id": 1,
        "question": "A bakery makes 48 cupcakes in the morning. They sell 3/4 of them by noon. In the afternoon, they bake another 30 cupcakes and sell 1/3 of the total cupcakes they currently have. How many cupcakes are left at the end of the day?",
        "ground_truth": 28
    },
    {
        "id": 2,
        "question": "A train leaves Town A at 2:00 PM traveling at 60 mph towards Town B. Another train leaves Town B at 3:00 PM traveling at 80 mph towards Town A. The distance between Town A and Town B is 270 miles. At what time (in PM, just write the hour as integer, e.g. 4) will they meet?",
        "ground_truth": 4
    },
    {
        "id": 3,
        "question": "Amir has 5 shirts, 3 pairs of pants, and 2 pairs of shoes. However, 1 of the shirts does not match with 1 of the pants (this specific combination cannot be worn). How many total unique complete outfits (1 shirt, 1 pants, 1 shoes) can he make?",
        "ground_truth": 28
    },
    {
        "id": 4,
        "question": "A snail is at the bottom of a 20-foot well. Each day it climbs up 5 feet, but each night it slides down 2 feet. On which day will the snail finally reach the top of the well?",
        "ground_truth": 6
    }
]

df_results = pd.DataFrame(questions)


In [12]:
# ========Helper Functions =========
def extract_final_number(text: str):
    """extract answer"""
    matches = re.findall(r"[-+]?\d*\.?\d+", text.replace(",", ""))
    if matches:
        val = matches[-1]
        return int(float(val)) if float(val).is_integer() else float(val)
    return None


In [13]:
# ========Direct Prompting (Zero-shot Direct) =========
direct_preds = []
direct_tokens = []

for q in questions:
    prompt = (
        f"Question: {q['question']}\n\n"
        "Provide only the final numerical answer. Do not include any explanations, units, or extra text."
    )
    res = complete_gap(prompt, temperature=0)
    answer_text = res.choices[0].message.content.strip()

    direct_preds.append(extract_final_number(answer_text))
    direct_tokens.append(res.usage.total_tokens)

df_results["Direct_Pred"] = direct_preds
df_results["Direct_Tokens"] = direct_tokens
print("Direct Prompting Done!")


Direct Prompting Done!


In [14]:
# ========Chain-of-Thought (Zero-shot CoT) =========
cot_preds = []
cot_tokens = []
cot_reasonings = []

for q in questions:
    prompt = (
        f"Question: {q['question']}\n\n"
        "Let's think step by step to find the correct solution. "
        "At the very end of your response, write the final numerical answer clearly on a new line in this format: 'Final Answer: <number>'."
    )
    res = complete_gap(prompt, temperature=0)
    raw_content = res.choices[0].message.content.strip()

    cot_reasonings.append(raw_content)
    cot_preds.append(extract_final_number(raw_content))
    cot_tokens.append(res.usage.total_tokens)

df_results["CoT_Pred"] = cot_preds
df_results["CoT_Tokens"] = cot_tokens
print("Zero-shot CoT Done!")


Zero-shot CoT Done!


In [15]:
# ========Self-Consistency (k=3 samples, temp=0.7) =========
K = 3
sc_final_preds = []
sc_total_tokens = []

for q in questions:
    prompt = (
        f"Question: {q['question']}\n\n"
        "Let's think step by step to solve this. "
        "At the very end, write the final number in the format: 'Final Answer: <number>'."
    )

    sample_answers = []
    task_tokens = 0

    for _ in range(K):
        res = complete_gap(prompt, temperature=0.7)
        ans = extract_final_number(res.choices[0].message.content)
        if ans is not None:
            sample_answers.append(ans)
        task_tokens += res.usage.total_tokens

    if sample_answers:
        majority_vote = Counter(sample_answers).most_common(1)[0][0]
    else:
        majority_vote = None

    sc_final_preds.append(majority_vote)
    sc_total_tokens.append(task_tokens)

df_results["SC_Pred"] = sc_final_preds
df_results["SC_Tokens"] = sc_total_tokens
print("Self-Consistency Done!")


Self-Consistency Done!


In [16]:
# ========Evaluation & Metrics =========
acc_direct = (df_results["Direct_Pred"] == df_results["ground_truth"]).mean() * 100
acc_cot = (df_results["CoT_Pred"] == df_results["ground_truth"]).mean() * 100
acc_sc = (df_results["SC_Pred"] == df_results["ground_truth"]).mean() * 100

avg_tokens_direct = df_results["Direct_Tokens"].mean()
avg_tokens_cot = df_results["CoT_Tokens"].mean()
avg_tokens_sc = df_results["SC_Tokens"].mean()

summary_metrics = pd.DataFrame({
    "Method": ["Direct", "Zero-shot CoT", f"Self-Consistency (k={K})"],
    "Accuracy (%)": [acc_direct, acc_cot, acc_sc],
    "Avg Tokens": [avg_tokens_direct, avg_tokens_cot, avg_tokens_sc]
})

print("=== Summary Comparison ===")
display(summary_metrics)

print("\n=== Detailed Predictions ===")
display(df_results[["id", "ground_truth", "Direct_Pred", "CoT_Pred", "SC_Pred"]])

summary_metrics.to_csv(root_path / "report" /"ImgPlt"/"02_summery_metrics.csv", index=False)
df_results.to_csv(root_path / "report" /"ImgPlt"/"02_cot_self_consistency.csv", index=False)

=== Summary Comparison ===


,Method,Accuracy (%),Avg Tokens
0,Direct,0.0,90.25
1,Zero-shot CoT,100.0,449.50
2,Self-Consistency (k=3),100.0,1454.00



=== Detailed Predictions ===


,id,ground_truth,Direct_Pred,CoT_Pred,SC_Pred
0,1,28,26,28,28
1,2,4,5,4,4
2,3,28,30,28,28
3,4,6,18,6,6
